# 06 - Figures for the report

Run the cells top to bottom, one at a time. Each figure has its own cell, so you can change a colour, a label or a size and re-run only that cell.

- Every figure is saved to `report/figures/` as PNG (for checking) and PDF (for the report).
- All drawdowns are the **true peak-to-trough drawdown**. The old `max_drawdown` column (a range ratio) is never used.
- Test period: 2022-07-01 to 2023-12-15, six trained seeds (seeds 4 and 5 share a training seed), hold interval N = 20 unless a figure says otherwise.

## 0. Setup
Paths, column names, colours and labels. Change them here and every figure follows.

In [ ]:
import os, pathlib
if pathlib.Path.cwd().name == "notebooks":      # run from the repo root so the paths below work
    os.chdir("..")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("report/figures"); OUT.mkdir(parents=True, exist_ok=True)
LOGS, TABLES = Path("results/logs"), Path("report/tables")

# column names in the equity-curve and results files
DATE, STRAT, SEED, VALUE = "date", "strategy", "train_seed", "portfolio_value"
RET, SHARPE, DD = "total_return", "sharpe_ratio", "max_drawdown_true"

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

COL = {"buy_and_hold": "#999999",
       "equal_weight_daily_rebalanced": "#444444",
       "MARL-no-coordinator": "#9467bd",
       "price-agent-only": "#bcbd22",
       "sentiment-agent-only": "#e377c2",
       "price-only-hold20": "#8c6d31",
       "sentiment-only-hold20": "#c2185b",
       "MARL-average-hold20": "#1f77b4",
       "price-static-hold20": "#8ab17d",
       "sentiment-static-hold20": "#2a9d8f",
       "MARL+LLM": "#d95f02"}

LABEL = {"buy_and_hold": "Buy and hold",
         "equal_weight_daily_rebalanced": "Equal-weight benchmark",
         "MARL-no-coordinator": "Agent average, daily",
         "price-agent-only": "Price agent, daily",
         "sentiment-agent-only": "Sentiment agent, daily",
         "price-only-hold20": "Price agent, 20-day hold",
         "sentiment-only-hold20": "Sentiment agent, 20-day hold",
         "MARL-average-hold20": "Agent average, 20-day hold",
         "price-static-hold20": "Static allocation (price agent's mean weights)",
         "sentiment-static-hold20": "Static allocation (sentiment agent's mean weights)",
         "MARL+LLM": "Agents + LLM coordinator, 20-day hold"}

FALL = (pd.Timestamp("2022-08-18"), pd.Timestamp("2022-09-30"))   # benchmark's peak to first trough

def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png"); fig.savefig(OUT / f"{name}.pdf")
    print("saved", OUT / f"{name}.png")

## 1. Load the equity curves
The non-LLM strategies come from `dd_test` and `static_control` (identical where they overlap). The MARL+LLM curves come from the replay of the `final3` run.

In [ ]:
EQ_FILES = [LOGS / "dd_test/equity_curves.csv",
            LOGS / "static_control/equity_curves.csv",
            LOGS / "replay_final3/equity_curves.csv"]

eq = pd.concat([pd.read_csv(f, parse_dates=[DATE]) for f in EQ_FILES if f.exists()])
eq = eq.drop_duplicates([STRAT, SEED, DATE])          # shared strategies appear in two folders
print(eq.groupby(STRAT)[SEED].nunique().rename("seeds (0 = single run)"))
print("days:", eq[DATE].nunique(), "|", eq[DATE].min().date(), "to", eq[DATE].max().date())

## Figure 1. Equity curves
Mean over the six seeds, with a band from the lowest to the highest seed. Change `SHOW` to pick the strategies, and `BANDS` to pick which ones get a band.

In [ ]:
SHOW  = ["equal_weight_daily_rebalanced", "sentiment-static-hold20", "MARL-average-hold20", "MARL+LLM"]
BANDS = ["MARL-average-hold20", "MARL+LLM"]

fig, ax = plt.subplots(figsize=(8, 4))
for name in SHOW:
    d = eq[eq[STRAT] == name]
    g = d.groupby(DATE)[VALUE]
    ax.plot(g.mean().index, g.mean().values, color=COL[name], label=LABEL[name], lw=1.6)
    if name in BANDS and d[SEED].nunique() > 1:
        ax.fill_between(g.mean().index, g.min().values, g.max().values, color=COL[name], alpha=0.12, lw=0)
ax.axvspan(*FALL, color="grey", alpha=0.12)
ax.yaxis.set_major_formatter(lambda v, _: f"${v/1000:.0f}k")
ax.set_ylabel("Portfolio value")
ax.legend(frameon=False, fontsize=8)
save(fig, "fig1_equity"); plt.show()

## Figure 2. Drawdown curves
Drawdown = value / running peak - 1, computed per seed and then averaged. The table reports the mean of each seed's maximum, which is nearly the same here because the troughs line up.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name in SHOW:
    d = eq[eq[STRAT] == name].sort_values(DATE).copy()
    d["dd"] = d.groupby(SEED, dropna=False)[VALUE].transform(lambda v: v / v.cummax() - 1) * 100
    g = d.groupby(DATE)["dd"]
    ax.plot(g.mean().index, g.mean().values, color=COL[name], label=LABEL[name], lw=1.6)
    if name in BANDS and d[SEED].nunique() > 1:
        ax.fill_between(g.mean().index, g.min().values, g.max().values, color=COL[name], alpha=0.12, lw=0)
    print(f"{LABEL[name]:55s} lowest point of the mean curve: {g.mean().min():6.1f}%")
ax.axvspan(*FALL, color="grey", alpha=0.12)
ax.set_ylabel("Drawdown from running peak (%)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
save(fig, "fig2_drawdown"); plt.show()

## 2. Load the per-seed results
One row per strategy and seed. The first print shows the column names; if `RET`, `SHARPE` or `DD` in the setup cell do not match, fix them there.

In [ ]:
RES_FILES = [LOGS / "dd_test/test_period_results.csv",
             LOGS / "static_control/test_period_results.csv",
             LOGS / "replay_final3/test_period_results.csv"]
res = pd.concat([pd.read_csv(f) for f in RES_FILES if f.exists()]).drop_duplicates([STRAT, SEED])
print(res.columns.tolist())
print(res.groupby(STRAT)[[RET, SHARPE, DD]].mean().round(3))     # compare with report/tables/master_table.md

## Figure 3. Each strategy, seed by seed
Small dots are the six seeds, the diamond is the mean, and the dashed line is the equal-weight benchmark.

In [ ]:
ORDER = ["buy_and_hold", "equal_weight_daily_rebalanced", "MARL-no-coordinator",
         "price-only-hold20", "sentiment-only-hold20", "MARL-average-hold20",
         "price-static-hold20", "sentiment-static-hold20", "MARL+LLM"]
METRICS = [(RET, "Total return (%)", 100), (SHARPE, "Sharpe ratio", 1), (DD, "Peak-to-trough drawdown (%)", 100)]

ypos = {s: i for i, s in enumerate(ORDER[::-1])}
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True)
for ax, (m, lab, k) in zip(axes, METRICS):
    ref = res.loc[res[STRAT] == "equal_weight_daily_rebalanced", m].mean() * k
    ax.axvline(ref, color="#444444", lw=0.8, ls="--")
    for s in ORDER:
        v = res.loc[res[STRAT] == s, m] * k
        ax.scatter(v, [ypos[s]] * len(v), s=18, color=COL[s], alpha=0.45, zorder=2)
        ax.scatter(v.mean(), ypos[s], s=70, marker="D", color=COL[s], edgecolor="white", zorder=3)
    ax.set_xlabel(lab); ax.grid(axis="x", alpha=0.25)
axes[0].set_yticks(list(ypos.values()))
axes[0].set_yticklabels([LABEL[s] for s in ypos])
save(fig, "fig3_dots"); plt.show()

## Figure 4. Paired differences (forest plot)
Each row is one comparison between two strategies **on the same six seeds**. The dot is the mean difference and the line is the 95% bootstrap confidence interval. Orange means the interval excludes zero; grey means it does not. The small `k/6` is the number of seeds with a positive difference.

- Return and drawdown are in percentage points. For drawdown, a negative value means the first strategy had the shallower drawdown.
- The first printed line lists the comparison names found in the tables. If a name in `PICK` is spelled differently, the cell reports it as not found.
- These are exploratory comparisons with no multiple-comparison correction. Say so in the caption.

In [ ]:
pair = pd.concat([pd.read_csv(TABLES / f) for f in ["paired_differences_summary_truedd.csv",
                                                     "paired_llm_vs_static.csv",
                                                     "paired_static_control.csv"]], ignore_index=True)
print(sorted(pair["comparison"].unique()))            # the exact names available

PICK = ["MARL+LLM minus MARL-average-hold20",          # RQ2: coordinator against the mechanical average
        "MARL+LLM minus equal_weight_daily_rebalanced",   # RQ3: against the benchmark
        "MARL+LLM minus sentiment-static-hold20",      # RQ3: against a fixed allocation
        "MARL+LLM minus price-static-hold20",
        "sentiment-only-hold20 minus sentiment-static-hold20",   # do the agents beat their own average holdings?
        "price-only-hold20 minus price-static-hold20",
        "sentiment-only-hold20 minus price-only-hold20"]         # RQ1: sentiment against price
SHORT = {"MARL+LLM": "LLM", "MARL-average-hold20": "Average", "equal_weight_daily_rebalanced": "Equal-weight",
         "sentiment-only-hold20": "Sentiment agent", "price-only-hold20": "Price agent",
         "sentiment-static-hold20": "Sentiment static", "price-static-hold20": "Price static",
         "MARL-no-coordinator": "Average (daily)"}
def nice(c):
    a, b = c.split(" minus ")
    return f"{SHORT.get(a, a)} − {SHORT.get(b, b)}"

PANELS = [(RET, "Return difference (points)", 100), (SHARPE, "Sharpe difference", 1),
          (DD, "Drawdown difference (points)\nnegative = first one is shallower", 100)]
missing = [c for c in PICK if c not in set(pair["comparison"])]
if missing: print("not found, skipped:", missing)
rows = [c for c in PICK if c not in missing]
ypos = {c: i for i, c in enumerate(rows[::-1])}

fig, axes = plt.subplots(1, 3, figsize=(11.5, 0.5 * len(rows) + 1.6), sharey=True)
for ax, (m, lab, k) in zip(axes, PANELS):
    ax.axvline(0, color="black", lw=0.8)
    for c in rows:
        r = pair[(pair["comparison"] == c) & (pair["metric"] == m)]
        if r.empty: continue
        r = r.iloc[0]
        lo, hi, mid = r["boot_ci_low"] * k, r["boot_ci_high"] * k, r["mean"] * k
        clear = (lo > 0) or (hi < 0)                   # interval excludes zero
        col = "#d95f02" if clear else "#777777"
        ax.plot([lo, hi], [ypos[c]] * 2, color=col, lw=2, solid_capstyle="round")
        ax.scatter(mid, ypos[c], s=36, color=col, zorder=3)
        left = mid < 0                                   # label goes on the outer side of the interval
        ax.annotate(f"{int(r['n_positive'])}/{int(r['n_seeds'])}", (lo if left else hi, ypos[c]),
                    xytext=(-5 if left else 5, 0), textcoords="offset points", va="center",
                    ha="right" if left else "left", fontsize=7, color="#555555")
    ax.set_xlabel(lab); ax.grid(axis="x", alpha=0.25); ax.margins(x=0.12)
axes[0].set_yticks(list(ypos.values())); axes[0].set_yticklabels([nice(c) for c in ypos])
save(fig, "fig4_paired_differences"); plt.show()

## Figure 5. Transaction-cost sensitivity
Mean return against the cost rate, with the standard deviation over seeds as error bars. Only return and Sharpe are used from this table, because its drawdown column is the old range ratio.

In [ ]:
cost = pd.read_csv(TABLES / "sensitivity_cost.csv")
print(cost.columns.tolist())
SHOW_COST = ["equal_weight_daily_rebalanced", "MARL-no-coordinator", "sentiment-agent-only",
             "MARL-average-hold20", "sentiment-only-hold20"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (m, s, lab, k) in zip(axes, [("mean_return", "std_return", "Total return (%)", 100),
                                     ("mean_sharpe", "std_sharpe", "Sharpe ratio", 1)]):
    for i, name in enumerate(SHOW_COST):
        d = cost[cost[STRAT] == name].sort_values("cost_rate")
        dodge = (i - (len(SHOW_COST) - 1) / 2) * 0.003          # small sideways shift so error bars do not overlap
        ax.errorbar(d["cost_rate"] * 100 + dodge, d[m] * k, yerr=d[s].fillna(0) * k, color=COL[name],
                    label=LABEL[name], marker="o", ms=4, lw=1.4, capsize=2)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xticks(sorted(cost["cost_rate"].unique() * 100))
    ax.set_xlabel("Transaction cost per unit of turnover (%)"); ax.set_ylabel(lab)
axes[0].legend(frameon=False, fontsize=8)
save(fig, "fig5_cost_sensitivity"); plt.show()

## Figure 6. Hold-interval sensitivity
The hold controls keep the label `hold20` in the file; the `hold_interval` column gives the real N. The MARL+LLM points come from the three LLM runs (N = 10, 20, 40), one run each.

In [ ]:
hold = pd.read_csv(TABLES / "sensitivity_hold.csv")
SHOW_HOLD = ["price-only-hold20", "sentiment-only-hold20", "MARL-average-hold20"]
HOLD_LABEL = {"price-only-hold20": "Price agent, hold N", "sentiment-only-hold20": "Sentiment agent, hold N",
              "MARL-average-hold20": "Agent average, hold N"}

llm = []                                       # MARL+LLM summary row from each interval run
for n, folder in [(10, "llm_cf10"), (20, "final3"), (40, "llm_cf40")]:
    f = LOGS / folder / "test_period_summary.csv"
    if f.exists():
        row = pd.read_csv(f).query("strategy == 'MARL+LLM'")
        if len(row): llm.append({"N": n, **row.iloc[0].to_dict()})
llm = pd.DataFrame(llm)
print(llm[["N", "mean_return", "mean_sharpe"]] if len(llm) else "no LLM summary files found")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (m, s, lab, k) in zip(axes, [("mean_return", "std_return", "Total return (%)", 100),
                                     ("mean_sharpe", "std_sharpe", "Sharpe ratio", 1)]):
    for i, name in enumerate(SHOW_HOLD):
        d = hold[hold[STRAT] == name].sort_values("hold_interval")
        dodge = 1 + (i - 1) * 0.04                               # small sideways shift so error bars do not overlap
        ax.errorbar(d["hold_interval"] * dodge, d[m] * k, yerr=d[s] * k, color=COL[name], label=HOLD_LABEL[name],
                    marker="o", ms=4, lw=1.4, capsize=2)
    ew = hold.loc[hold[STRAT] == "equal_weight_daily_rebalanced", m].iloc[0] * k
    ax.axhline(ew, color=COL["equal_weight_daily_rebalanced"], ls="--", lw=1, label="Equal-weight benchmark")
    if len(llm):
        ax.errorbar(llm["N"] * 1.08, llm[m].astype(float) * k, yerr=llm[s].astype(float) * k, color=COL["MARL+LLM"],
                    label="Agents + LLM coordinator", marker="D", ms=6, lw=0, elinewidth=1.2, capsize=2)
    ax.set_xscale("log"); ax.set_xticks([5, 10, 20, 40]); ax.set_xticklabels([5, 10, 20, 40])
    ax.minorticks_off()
    ax.set_xlabel("Hold interval N (trading days)"); ax.set_ylabel(lab)
axes[0].legend(frameon=False, fontsize=8)
save(fig, "fig6_hold_sensitivity"); plt.show()

## Figure 6b. MARL+LLM against the agent average at N = 10, 20 and 40
Return, Sharpe and **true** drawdown for the two strategies at each interval (mean and standard deviation over six seeds). This is the table to use for drawdown across intervals; Figure 6 cannot show it, because the sensitivity tables hold only the old range ratio.

In [ ]:
li = pd.read_csv(TABLES / "llm_across_intervals_truedd.csv")
P6 = [("total_return", "Total return (%)", 100), ("sharpe_ratio", "Sharpe ratio", 1),
      ("max_drawdown_true", "Peak-to-trough drawdown (%)", 100)]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (m, lab, k) in zip(axes, P6):
    for i, name in enumerate(["MARL-average-hold20", "MARL+LLM"]):
        d = li[li[STRAT] == name].sort_values("N")
        ax.errorbar(d["N"] * (0.97 if i == 0 else 1.03), d[f"{m}_mean"] * k, yerr=d[f"{m}_std"] * k,
                    color=COL[name], label=LABEL[name].replace(", 20-day hold", ""), marker="o", ms=4, lw=1.4, capsize=2)
    ax.set_xscale("log"); ax.set_xticks([10, 20, 40]); ax.set_xticklabels([10, 20, 40]); ax.minorticks_off()
    ax.set_xlabel("Hold interval N (trading days)"); ax.set_ylabel(lab)
axes[0].legend(frameon=False, fontsize=8)
save(fig, "fig6b_llm_across_intervals"); plt.show()

## Figure 7. Training period against test period (overfitting)
Hollow marker = training period (in-sample), filled marker = test period. The two periods have different lengths (958 and 368 trading days), so the return is **annualised** to make them comparable; the Sharpe ratio is already annualised. A long line from a high hollow marker to a low filled one means the strategy looked good on the data it was trained on and did not carry over.

In [ ]:
from matplotlib.lines import Line2D
tt = pd.read_csv(TABLES / "train_vs_test_truedd.csv").set_index(STRAT)
TRAIN_DAYS, TEST_DAYS = 958, 368
ORDER7 = ["equal_weight_daily_rebalanced", "MARL-no-coordinator", "price-agent-only", "sentiment-agent-only",
          "price-only-hold20", "sentiment-only-hold20", "MARL-average-hold20"]
ann = lambda r, days: ((1 + r) ** (252 / days) - 1) * 100       # annualised return in %

ypos = {s: i for i, s in enumerate(ORDER7[::-1])}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for s in ORDER7:
    pts = [(ann(tt.loc[s, "train_mean_return"], TRAIN_DAYS), ann(tt.loc[s, "test_mean_return"], TEST_DAYS)),
           (tt.loc[s, "train_mean_sharpe"], tt.loc[s, "test_mean_sharpe"])]
    for ax, (tr, te) in zip(axes, pts):
        ax.plot([tr, te], [ypos[s]] * 2, color=COL[s], lw=1.5, alpha=0.6)
        ax.scatter(tr, ypos[s], s=45, facecolor="white", edgecolor=COL[s], zorder=3)
        ax.scatter(te, ypos[s], s=45, color=COL[s], zorder=3)
for ax, lab in zip(axes, ["Annualised return (%)", "Sharpe ratio"]):
    ax.set_xlabel(lab); ax.grid(axis="x", alpha=0.25)
axes[0].set_yticks(list(ypos.values())); axes[0].set_yticklabels([LABEL[s] for s in ypos])
axes[1].legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec="#444444", label="Training period"),
                        Line2D([], [], marker="o", ls="", color="#444444", label="Test period")],
               frameon=False, fontsize=8)
save(fig, "fig7_train_vs_test"); plt.show()

## Figure 8. What the coordinator changes
Mean portfolio weight per asset over the 114 coordinator calls of the `final3` run: the plain average of the two agents next to the weights the LLM chose. Bars are the overall mean and dots are the six seeds. The first printed line shows the scopes and metrics in the table; the cell picks the metric that has both an `llm` and an `average` value for every asset.

In [ ]:
cb = pd.read_csv(TABLES / "coordinator_behaviour.csv")
print(cb["scope"].unique(), cb["metric"].unique())
ASSETS = ["INTC", "AMD", "GS", "CVX", "SBUX", "cash"]

w = cb.dropna(subset=["llm", "average"]); w = w[w["asset"].isin(ASSETS)]
metric = w.groupby("metric")["asset"].nunique().idxmax(); w = w[w["metric"] == metric]
print("metric used:", metric)
is_seed = w["scope"].astype(str).str.startswith("seed")
seeds = w[is_seed]
overall = w[~is_seed].groupby("asset")[["llm", "average"]].mean()
if overall.empty: overall = seeds.groupby("asset")[["llm", "average"]].mean()
overall = overall.reindex(ASSETS)

x = np.arange(len(ASSETS)); bw = 0.38
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for off, colname, name in [(-bw / 2, "average", "MARL-average-hold20"), (bw / 2, "llm", "MARL+LLM")]:
    ax.bar(x + off, overall[colname] * 100, bw, color=COL[name], alpha=0.75,
           label="Agent average" if colname == "average" else "LLM coordinator")
    for i, a in enumerate(ASSETS):
        v = seeds.loc[seeds["asset"] == a, colname] * 100
        ax.scatter([x[i] + off] * len(v), v, s=10, color="black", alpha=0.45, zorder=3)
ax.set_xticks(x); ax.set_xticklabels(ASSETS); ax.set_ylabel("Mean portfolio weight (%)")
ax.legend(frameon=False, fontsize=8)
save(fig, "fig8_coordinator_weights"); plt.show()
print((overall * 100).round(1))

## Figure 9a. What each agent relies on (SHAP group shares)
For each seed, the share of the agent's total attribution that falls on its own signal features and on the portfolio and cash state. The share is the sum of per-feature mean |SHAP| in the group divided by the agent's total. SHAP shows what a policy uses, not whether using it helps.

In [ ]:
sh = pd.read_csv(TABLES / "shap_group_shares.csv")
print(sh.groupby(["agent", "group"])["share"].agg(["mean", "std"]).round(3))
SIGNAL = {"price": ("price_indicators", "Price indicators (50 features)", "#8c6d31"),
          "sentiment": ("sentiment_features", "Sentiment features (20 features)", "#c2185b")}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, agent in zip(axes, ["price", "sentiment"]):
    grp, lab, c = SIGNAL[agent]
    d = sh[sh["agent"] == agent].pivot(index="seed", columns="group", values="share").sort_index()
    ax.bar(d.index, d[grp] * 100, color=c, alpha=0.8, label=lab)
    ax.bar(d.index, d["portfolio_cash_state"] * 100, bottom=d[grp] * 100, color="#bbbbbb",
           label="Portfolio and cash state (7 features)")
    ax.set_title(f"{agent.capitalize()} agent", fontsize=9); ax.set_xlabel("Training seed")
    ax.set_xticks(list(d.index)); ax.legend(frameon=False, fontsize=7, loc="lower center")
axes[0].set_ylabel("Share of total attribution (%)")
save(fig, "fig9a_shap_shares"); plt.show()

## Figure 9b. Does scrambling the inputs hurt? (input ablation)
Change in the Sharpe ratio on the test period when an agent's own inputs are shuffled across days, compared with the unmodified run. The dot is the mean over 20 shuffles and the bar is the standard deviation over those shuffles. The cross is the run with `portfolio_norm` held at 1. A dot clearly below zero means the agent's result depends on the real content of that input.

In [ ]:
ab = pd.read_csv(TABLES / "ablation_inputs.csv")
print(ab.groupby([STRAT, "intervention"]).size().unstack(fill_value=0))
PAN = [("price-only-hold20", "shuffle_price", "Price agent, 20-day hold: price inputs shuffled"),
       ("sentiment-only-hold20", "shuffle_sentiment", "Sentiment agent, 20-day hold: sentiment inputs shuffled")]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (s, iv, title) in zip(axes, PAN):
    d = ab[(ab[STRAT] == s) & (ab["intervention"] == iv)].sort_values("train_seed")
    n = ab[(ab[STRAT] == s) & (ab["intervention"] == "norm_const")].sort_values("train_seed")
    ax.axhline(0, color="black", lw=0.8)
    ax.errorbar(d["train_seed"], d["delta_sharpe_mean"], yerr=d["delta_sharpe_std"], fmt="o", color=COL[s],
                capsize=3, label="Inputs shuffled (mean and std over 20 draws)")
    ax.scatter(n["train_seed"] + 0.18, n["delta_sharpe_mean"], marker="x", color="#555555",
               label="portfolio_norm held at 1")
    ax.set_title(title, fontsize=9); ax.set_xlabel("Training seed"); ax.set_xticks(sorted(d["train_seed"].unique()))
axes[0].set_ylabel("Change in Sharpe ratio"); axes[0].legend(frameon=False, fontsize=7)
save(fig, "fig9b_ablation"); plt.show()

## Figure 10. The stress episode (benchmark's fall and recovery)
Same dates for every strategy, set from the equal-weight benchmark only: fall phase 2022-08-18 to 2022-09-30, recovery phase to 2022-11-30. Dots are seeds, the diamond is the mean, and the dashed line is the benchmark. The benchmark's trough is a near-tie with 2022-10-11, so the drawdown inside the episode (third panel) also captures the second low. The first print shows the sources in the table; `LLM_SOURCE` picks the headline MARL+LLM run.

In [ ]:
st = pd.read_csv(TABLES / "stress_episode.csv")
print(st.groupby(["source", "interval_N"])[STRAT].nunique())
LLM_SOURCE = "replay_final3"
COL.setdefault("MARL-average-hold20-cash5.5", "#6baed6")
LABEL.setdefault("MARL-average-hold20-cash5.5", "Agent average + 5.5% cash, 20-day hold")

is_llm = st[STRAT] == "MARL+LLM"
keep = st[(is_llm & (st["source"] == LLM_SOURCE)) | (~is_llm & (st["interval_N"] == 20))]
keep = keep.drop_duplicates([STRAT, SEED])
ORDER10 = ["equal_weight_daily_rebalanced", "MARL-average-hold20", "MARL-average-hold20-cash5.5", "MARL+LLM"]
M10 = [("fall_return", "Fall phase return (%)"), ("recovery_return", "Recovery phase return (%)"),
       ("episode_drawdown_true", "Drawdown inside the episode (%)")]

ypos = {s: i for i, s in enumerate(ORDER10[::-1])}
fig, axes = plt.subplots(1, 3, figsize=(11, 2.9), sharey=True)
for ax, (m, lab) in zip(axes, M10):
    ax.axvline(keep.loc[keep[STRAT] == "equal_weight_daily_rebalanced", m].mean() * 100, color="#444444", lw=0.8, ls="--")
    for s in ORDER10:
        v = keep.loc[keep[STRAT] == s, m] * 100
        if v.empty: print("no rows for", s); continue
        ax.scatter(v, [ypos[s]] * len(v), s=18, color=COL[s], alpha=0.45, zorder=2)
        ax.scatter(v.mean(), ypos[s], s=70, marker="D", color=COL[s], edgecolor="white", zorder=3)
    ax.set_xlabel(lab); ax.grid(axis="x", alpha=0.25)
axes[0].set_yticks(list(ypos.values())); axes[0].set_yticklabels([LABEL[s] for s in ypos])
save(fig, "fig10_stress_episode"); plt.show()

## Before a figure goes into the report
- The caption states the test period (2022-07-01 to 2023-12-15), six trained seeds (seeds 4 and 5 share a training seed), the hold interval, and that drawdown is peak-to-trough.
- Numbers quoted in the text come from `report/tables/master_table.md` and the `_truedd` tables, not from reading a figure.
- Comparisons added after the first results (hold controls, cash controls, interval runs, stress episode, static controls, ablation) are described as exploratory.